# Mini zbiór testowy

Wspólny mały zbiór testowy do porównania modeli LLM z pozostałymi metodami. Pełny test ID (13 k wiadomości) i korpus OOD
(6.1 k) są zbyt duże, aby lokalny LLM przetworzył je kilka razy w rozsądnym czasie.

- **ID:** wiadomości ze zbioru testowego podziału z ziarnem 42 (`make_split`), czyli rozłączne z danymi uczącymi wszystkich
  modeli. Po 300 z każdego źródła. W źródłach TREC po 150 spamu i 150 hamu.
- **OOD:** po 300 z FERB i Zero-Day (po 150 phishingu i hamu), całe NEBULON (145) i KD (60).
- **Losowanie:** prosta próba bez zwracania z ziarnem 42 w każdej warstwie (źródło, klasa). Gdy warstwa jest mniejsza niż
  wymagana liczba, brana jest w całości.
- **Wynik:** `data/mini_test.parquet` z identyfikatorem `id` (`raw_key` dla ID, `uid` dla OOD), etykietami i tekstem.
  Zbiór jest zbalansowany, więc metryki zależne od udziału klasy pozytywnej (precision, AUCPR) nie są porównywalne
  z pełnym testem.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "notebooks"))
from experiment_utils import ID_SOURCES, OOD_SOURCES, load_corpora

RESULTS = ROOT / "results" / "mini_test_set"
TABLES = RESULTS / "tables"
TABLES.mkdir(parents=True, exist_ok=True)
OUTPUT = ROOT / "data" / "mini_test.parquet"
SEED = 42
PER_SOURCE = 300
COLUMNS = ["id", "korpus", "source", "kind", "label_spam_0", "label_spam_1", "cluster", "text"]

## Losowanie

In [2]:
in_domain, out_of_domain = load_corpora()
test = in_domain[in_domain.split == "test"].rename(columns={"raw_key": "id"}).assign(korpus="ID")
ood = out_of_domain.rename(columns={"uid": "id"}).assign(korpus="OOD")
ood["id"] = ood["id"].astype(str)


def sample(frame):
    """Po PER_SOURCE wiadomości na źródło, po równo na klasy obecne w źródle."""
    kinds = sorted(frame.kind.unique())
    per_kind = PER_SOURCE // len(kinds)
    parts = [group.sample(min(len(group), per_kind), random_state=SEED) for _, group in frame.groupby("kind")]
    return pd.concat(parts)


mini = pd.concat(
    [sample(test[test.source == name]) for name in ID_SOURCES] + [sample(ood[ood.source == name]) for name in OOD_SOURCES]
)[COLUMNS].reset_index(drop=True)
assert mini.id.is_unique
mini.to_parquet(OUTPUT, index=False)
print(OUTPUT, len(mini))

/home/ubuntu/phishing-thesis/data/mini_test.parquet 2305


## Skład zbioru

In [3]:
composition = mini.pivot_table(index=["korpus", "source"], columns="kind", values="id", aggfunc="size", fill_value=0)
composition["razem"] = composition.sum(axis=1)
composition["udział_spam_1_%"] = mini.groupby(["korpus", "source"]).label_spam_1.mean() * 100
composition.to_csv(TABLES / "sklad_mini_zbioru.csv")
display(composition)
print(mini.groupby("korpus").agg(n=("id", "size"), pozytywne_spam_1=("label_spam_1", "sum"), mediana_znaków=("text", lambda t: t.str.len().median())))

kind             ham  phishing  spam  razem  udział_spam_1_%
korpus source                                               
ID     nazario     0       300     0    300       100.000000
       nigerian    0       300     0    300       100.000000
       trec05    150         0   150    300        50.000000
       trec06    150         0   150    300        50.000000
       trec07    150         0   150    300        50.000000
OOD    ferb      150       150     0    300        50.000000
       kd10000     5        55     0     60        91.666667
       nebulon    76        69     0    145        47.586207
       zeroday   150       150     0    300        50.000000

           n  pozytywne_spam_1  mediana_znaków
korpus                                        
ID      1500              1050           809.0
OOD      805               424           469.0
